# Chapter 2 -- The Agent Loop from Scratch (Practice)

Work through this notebook **after reading** `notes/ch02-agent-loop-from-scratch.md`. This chapter builds a real agent loop against Claude's native tool-use wire format (`tool_use` / `tool_result` blocks, `stop_reason`) -- the exact mechanics the notes describe in detail, using notes Section 11's own `read_file` / `word_count` task.

Three exercises below have a stub to fill in: the **budget guard**, **parallel tool-call handling**, and **JSONL trajectory logging**. Each is followed by a verification cell that runs against a scripted, deterministic fake client -- no API key needed to complete any exercise. A final optional section lets you run the exact same loop, unchanged, against a real Claude model.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline against a")
        print("scripted fake client -- this cell only matters for the optional")
        print("real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline against a scripted fake client.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The task and its two tools

The agent's job: read two text files and report their combined word count. Two tools are available -- `read_file(path)` and `word_count(text)` -- the exact pair from notes Section 11's dry-run.

In [ ]:
SAMPLE_DIR = Path.cwd() / "ch02_sample_files"
SAMPLE_DIR.mkdir(exist_ok=True)

(SAMPLE_DIR / "report.txt").write_text(
    "Quarterly revenue grew twelve percent driven by strong demand in the "
    "enterprise segment and continued expansion in international markets."
)
(SAMPLE_DIR / "business.txt").write_text(
    "The board approved a new capital allocation plan focused on research "
    "and disciplined cost management across every division this year."
)

print("Sample files created:")
for f in sorted(SAMPLE_DIR.iterdir()):
    print(f"  {f.name}: {len(f.read_text().split())} words")


def read_file(path: str) -> str:
    """Return the contents of a file at `path` (resolved inside the sample directory)."""
    target = SAMPLE_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"{path} not found in {SAMPLE_DIR.name}/")
    return target.read_text()


def word_count(text: str) -> str:
    """Return the number of whitespace-separated words in `text`, as a string."""
    return str(len(text.split()))


TOOL_DISPATCH = {
    "read_file": read_file,
    "word_count": word_count,
}

TOOL_SCHEMAS = [
    {
        "name": "read_file",
        "description": "Read the full contents of a text file by name. Call this before word_count if you need the file content.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "File name, e.g. \'report.txt\'"}},
            "required": ["path"],
        },
    },
    {
        "name": "word_count",
        "description": "Count the number of whitespace-separated words in a string of text you already have.",
        "input_schema": {
            "type": "object",
            "properties": {"text": {"type": "string", "description": "The text to count words in"}},
            "required": ["text"],
        },
    },
]


In [ ]:
def execute_tool_call(block) -> tuple:
    """
    Run one tool_use block through TOOL_DISPATCH.
    Returns (content_string, is_error) -- matches notes Section 6 exactly:
    exceptions become an error tool_result, never a crash.
    """
    fn = TOOL_DISPATCH.get(block.name)
    if fn is None:
        return f"Error: no such tool \'{block.name}\'", True
    try:
        result = fn(**block.input)
        return str(result), False
    except Exception as exc:
        return f"Error: {exc}", True


## Testing offline: a scripted fake client

Exercises 1-3 need something that behaves like a real model response (`.stop_reason`, `.content`, `.usage`) without spending money or depending on what a real model happens to do on any given run. `FakeClient` below plays back a fixed script of responses -- including a deliberately wrong file name in step 1, so you can watch the loop's error-handling and parallel-tool-call behavior exactly once, reliably, every time you run this notebook.

The script: **step 1** asks for two files in parallel -- `report.txt` (will succeed) and `business2.txt` (a wrong name -- will fail). **Step 2** retries with the corrected name. **Step 3** calls `word_count` on both files in parallel. **Step 4** completes naturally.

In [ ]:
from types import SimpleNamespace


def _text(text):
    return SimpleNamespace(type="text", text=text)


def _tool_use(id, name, input):
    return SimpleNamespace(type="tool_use", id=id, name=name, input=input)


def _response(stop_reason, content, input_tokens, output_tokens):
    return SimpleNamespace(
        stop_reason=stop_reason,
        content=content,
        usage=SimpleNamespace(input_tokens=input_tokens, output_tokens=output_tokens),
    )


_REPORT_TEXT = read_file("report.txt")
_BUSINESS_TEXT = read_file("business.txt")
_COMBINED_COUNT = len(_REPORT_TEXT.split()) + len(_BUSINESS_TEXT.split())

FAKE_SCRIPT = [
    _response(
        "tool_use",
        [
            _text("I will read both files."),
            _tool_use("toolu_1", "read_file", {"path": "report.txt"}),
            _tool_use("toolu_2", "read_file", {"path": "business2.txt"}),
        ],
        input_tokens=300, output_tokens=40,
    ),
    _response(
        "tool_use",
        [_tool_use("toolu_3", "read_file", {"path": "business.txt"})],
        input_tokens=420, output_tokens=15,
    ),
    _response(
        "tool_use",
        [
            _tool_use("toolu_4", "word_count", {"text": _REPORT_TEXT}),
            _tool_use("toolu_5", "word_count", {"text": _BUSINESS_TEXT}),
        ],
        input_tokens=900, output_tokens=60,
    ),
    _response(
        "end_turn",
        [_text(f"The combined word count is {_COMBINED_COUNT} words.")],
        input_tokens=980, output_tokens=12,
    ),
]


class FakeClient:
    """Plays back FAKE_SCRIPT one response per call, ignoring the actual request."""

    def __init__(self, script):
        self._script = list(script)
        self._calls = 0

    class _Messages:
        def __init__(self, outer):
            self._outer = outer

        def create(self, **kwargs):
            if self._outer._calls >= len(self._outer._script):
                raise RuntimeError("FakeClient script exhausted -- the loop did not stop in time")
            response = self._outer._script[self._outer._calls]
            self._outer._calls += 1
            return response

    @property
    def messages(self):
        return FakeClient._Messages(self)


print(f"Scripted trajectory ready: {len(FAKE_SCRIPT)} steps.")
print(f"Real combined word count (for later comparison): {_COMBINED_COUNT}")


## Exercises -- complete `run_agent_loop`

One function, three TODOs, matching notes Sections 5, 7, and 9:

1. **TODO 1 -- budget guard.** Before each request, if `cumulative_tokens` already exceeds `max_token_budget`, print a message and return early as `(None, step - 1, cumulative_tokens)`.
2. **TODO 2 -- parallel tool handling.** `tool_use_blocks` may hold more than one block. Build one `tool_result` dict per block and send **all** of them back in a **single** user turn -- never split them across messages.
3. **TODO 3 -- trajectory logging.** If `log_path` is given, append one JSON line per step (every step, including the final `end_turn`) recording `step`, `stop_reason`, `cumulative_tokens`, and `tool_calls_this_step`.

Everything else in the function (the request, the `stop_reason` branch, saving the assistant's full turn before executing tools) is given and already matches notes Sections 3 and 6.

In [ ]:
import json

MAX_TOKEN_BUDGET = 5000
MAX_STEPS = 10


def run_agent_loop(client, model, tools, messages, max_steps=MAX_STEPS,
                    max_token_budget=MAX_TOKEN_BUDGET, log_path=None):
    """
    The full agent loop from notes Sections 3, 5, 6, 7, 9 -- request, branch
    on stop_reason, execute every tool_use block, return all results in one
    turn, guard against runaway budgets, and log every step to JSONL.

    Returns: (final_text_or_None, steps_taken, cumulative_tokens)
    """
    cumulative_tokens = 0
    if log_path is not None and log_path.exists():
        log_path.unlink()  # start each run with a clean trajectory file

    for step in range(1, max_steps + 1):
        # --- TODO 1: BUDGET GUARD ---
        # If cumulative_tokens already exceeds max_token_budget, print a
        # clear message and return (None, step - 1, cumulative_tokens).
        pass  # TODO 1

        response = client.messages.create(
            model=model, max_tokens=1024, tools=tools, messages=messages,
        )
        cumulative_tokens += response.usage.input_tokens + response.usage.output_tokens
        print(f"STEP {step} | stop_reason={response.stop_reason} | "
              f"cumulative_tokens={cumulative_tokens}")

        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

        # --- TODO 3: TRAJECTORY LOGGING (every step, regardless of stop_reason) ---
        # If log_path is not None, append one JSON line recording: step,
        # stop_reason, cumulative_tokens, and len(tool_use_blocks).
        pass  # TODO 3

        if response.stop_reason == "end_turn":
            final_text = next((b.text for b in response.content if b.type == "text"), "")
            print(f"  Final answer: {final_text}")
            return final_text, step, cumulative_tokens

        # Save the assistant\'s FULL turn before executing anything (notes Section 3).
        messages.append({"role": "assistant", "content": response.content})

        results = []  # TODO 2: build one tool_result dict per block, append below
        for block in tool_use_blocks:
            content_str, is_error = execute_tool_call(block)
            status = "ERROR" if is_error else "ok"
            print(f"    tool_use {block.name}({block.input}) -> [{status}] "
                  f"{content_str[:60]}")
            # TODO 2: build the tool_result dict for this block and append it to `results`

        # TODO 2: send ALL of `results` back in a single user turn (notes Section 7)

    print(f"Hit max_steps ({max_steps}) without a natural stop.")
    return None, max_steps, cumulative_tokens


**Verification -- Exercise 1 (budget guard)**

Run with a deliberately tiny budget and confirm the loop stops *before* it could finish naturally, rather than crashing or running to the end regardless.

In [ ]:
fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count."}]
final_text, steps_taken, tokens = run_agent_loop(
    FakeClient(FAKE_SCRIPT), model="fake-model", tools=TOOL_SCHEMAS,
    messages=fresh_messages, max_token_budget=500,  # deliberately tiny
)

assert final_text is None, "Expected the budget guard to stop the loop before a natural end_turn."
assert steps_taken < len(FAKE_SCRIPT), f"Expected an early stop, got {steps_taken} steps."
print(f"PASS -- budget guard stopped the loop after {steps_taken} step(s), before it could finish naturally.")


**Verification -- Exercise 2 (parallel tool handling)**

Run to natural completion and inspect the message list: step 1's two `tool_use` blocks (one success, one error) must come back together in a single user turn.

In [ ]:
fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count."}]
final_text, steps_taken, tokens = run_agent_loop(
    FakeClient(FAKE_SCRIPT), model="fake-model", tools=TOOL_SCHEMAS,
    messages=fresh_messages, max_token_budget=10_000,
)

assert final_text is not None, "Expected the loop to complete naturally -- check that TODO 1 is not stopping it early."
assert steps_taken == 4, f"Expected exactly 4 steps, got {steps_taken}."

# fresh_messages[0] = initial user message
# fresh_messages[1] = assistant turn 1 (the 2 parallel read_file calls)
# fresh_messages[2] = the ONE user turn carrying BOTH tool_result blocks from step 1
step1_results_turn = fresh_messages[2]
assert step1_results_turn["role"] == "user"
assert len(step1_results_turn["content"]) == 2, (
    "Expected both step-1 tool_result blocks in a SINGLE user turn -- "
    "did you split them across two messages? (notes Section 7)"
)
statuses = {r["tool_use_id"]: r["is_error"] for r in step1_results_turn["content"]}
assert statuses["toolu_1"] is False, "read_file(\'report.txt\') should have succeeded."
assert statuses["toolu_2"] is True, "read_file(\'business2.txt\') should have failed -- that file does not exist."
print("PASS -- both parallel tool calls from step 1 executed and returned together, mixed success/error handled correctly.")


**Verification -- Exercise 3 (trajectory logging)**

Run once more with `log_path` set and confirm every step was written to disk as valid JSON.

In [ ]:
TRAJECTORY_PATH = Path.cwd() / "trajectory.jsonl"

fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count."}]
final_text, steps_taken, tokens = run_agent_loop(
    FakeClient(FAKE_SCRIPT), model="fake-model", tools=TOOL_SCHEMAS,
    messages=fresh_messages, max_token_budget=10_000, log_path=TRAJECTORY_PATH,
)

assert TRAJECTORY_PATH.exists(), "No trajectory.jsonl was written -- check TODO 3."
lines = TRAJECTORY_PATH.read_text().strip().splitlines()
assert len(lines) == steps_taken, f"Expected {steps_taken} logged steps, found {len(lines)}."

first_record = json.loads(lines[0])
assert first_record["step"] == 1
assert first_record["stop_reason"] == "tool_use"
assert first_record["tool_calls_this_step"] == 2

print(f"PASS -- {len(lines)} steps logged to {TRAJECTORY_PATH.name}, each record well-formed.")
for line in lines:
    print(" ", line)


### Optional -- run the exact same loop against a real Claude model

`run_agent_loop` never touched `FakeClient`'s internals directly -- it only called `.messages.create(model=..., max_tokens=..., tools=..., messages=...)` and read `.stop_reason` / `.content` / `.usage` off the result. A real `AnthropicBedrockMantle` client (Claude Sonnet, via AWS Bedrock) exposes the identical shape, so the same function runs unchanged against a live model. Flip `RUN_REAL_AGENT_DEMO` to `True` to try it.

In [ ]:
RUN_REAL_AGENT_DEMO = False


def run_real_agent_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real agent demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    real_messages = [{
        "role": "user",
        "content": "Read report.txt and business.txt, then tell me the combined word count. Use the tools -- do not guess.",
    }]
    # A real client raises the SDK's own exception types directly (e.g.
    # AuthenticationError) -- unlike FakeClient, nothing here catches or
    # reshapes those, so wrap the run itself rather than letting a raw
    # traceback surface.
    try:
        final_text, steps_taken, tokens = run_agent_loop(
            real_client, model=MODEL_NAME, tools=TOOL_SCHEMAS,
            messages=real_messages, max_token_budget=20_000,
            log_path=Path.cwd() / "trajectory_real.jsonl",
        )
    except Exception as exc:
        print(f"Real agent run failed: {type(exc).__name__}: {exc}")
        return

    print()
    print(f"Real run finished in {steps_taken} step(s), {tokens} cumulative tokens.")
    print(f"Real combined word count for comparison: {_COMBINED_COUNT}")


if RUN_REAL_AGENT_DEMO:
    run_real_agent_demo()
else:
    print("RUN_REAL_AGENT_DEMO is False -- running in offline/scripted mode only.")
    print("Flip it to True to run this exact loop against a real Claude model via Bedrock.")


## Key Takeaways

You now have a real, self-correcting, parallel-tool-aware, budget-guarded, self-logging agent loop -- built from a request/response cycle, a `stop_reason` branch, and a dispatch table. It is not a simplification of what a framework does internally; it is structurally the same loop, and every later chapter in this folder attaches to it at a specific, named seam (memory, verification, permissions, persistence, evals).

**Connection forward:** Chapter 3 goes back to the tool definitions this chapter treated as given -- naming, schema shape, granularity, error-message wording -- and asks what actually makes one tool design good and another one quietly sabotage the model calling it.